# Model Training & Calibration

Train prediction models on collected data, evaluate calibration,
and analyze edge distributions.

In [ ]:
import sys
sys.path.insert(0, '../src')

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from decimal import Decimal
from datetime import datetime, timezone

from moneygone.models.calibration import IsotonicCalibrator, PlattCalibrator
from moneygone.models.evaluation import ModelEvaluator
from moneygone.signals.fees import KalshiFeeCalculator
from moneygone.signals.edge import EdgeCalculator

DATA_DIR = Path('../data')
fee_calc = KalshiFeeCalculator()

## Load Historical Market Data

In [ ]:
# Load market states with outcomes for settled markets
MARKET_DB = DATA_DIR / 'market_data.duckdb'
con = duckdb.connect(str(MARKET_DB), read_only=True)

# Get settled markets with price history
df_settled = con.execute("""
    WITH last_state AS (
        SELECT ticker, title, category, result, last_price,
               yes_bid, yes_ask, volume, open_interest,
               recorded_at,
               ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY recorded_at DESC) as rn
        FROM market_states
        WHERE result IN ('yes', 'no')
    )
    SELECT * FROM last_state WHERE rn = 1
""").df()

print(f'Settled markets with data: {len(df_settled)}')
if len(df_settled) > 0:
    df_settled['outcome'] = (df_settled['result'] == 'yes').astype(int)
    df_settled.head()

## Market Price Calibration Analysis

Are Kalshi market prices well-calibrated? Do markets priced at 60c resolve YES ~60% of the time?

In [ ]:
if len(df_settled) >= 20:
    # Bin markets by their last traded price
    df_settled['price_bin'] = pd.cut(df_settled['last_price'], bins=10)
    calibration = df_settled.groupby('price_bin').agg(
        mean_price=('last_price', 'mean'),
        mean_outcome=('outcome', 'mean'),
        count=('outcome', 'count')
    ).reset_index()
    
    print('Market Price Calibration:')
    print('Price Bin | Avg Price | Actual % YES | Count')
    for _, row in calibration.iterrows():
        print(f"  {row['price_bin']} | {row['mean_price']:.3f} | {row['mean_outcome']:.3f} | {row['count']}")
else:
    print(f'Need more settled markets for calibration analysis (have {len(df_settled)}, need 20+)')

## Sportsbook Model Training

In [ ]:
# Load sportsbook lines for model training
COLLECTOR_DB = DATA_DIR / 'collector.duckdb'
if COLLECTOR_DB.exists():
    con2 = duckdb.connect(str(COLLECTOR_DB), read_only=True)
    
    df_lines = con2.execute("""
        SELECT league, game_id, bookmaker, market_type,
               home_team, away_team, home_odds, away_odds,
               captured_at
        FROM sportsbook_game_lines
        ORDER BY captured_at
    """).df()
    
    print(f'Total sportsbook records: {len(df_lines)}')
    print(f'Games covered: {df_lines["game_id"].nunique()}')
    print(f'\nBy bookmaker:')
    print(df_lines['bookmaker'].value_counts())
    
    # Compute implied probabilities from odds
    if len(df_lines) > 0:
        pinnacle = df_lines[df_lines['bookmaker'] == 'pinnacle'].copy()
        if len(pinnacle) > 0:
            # American odds to implied probability
            def odds_to_prob(odds):
                if odds is None or pd.isna(odds):
                    return None
                odds = float(odds)
                if odds > 0:
                    return 100 / (odds + 100)
                else:
                    return abs(odds) / (abs(odds) + 100)
            
            pinnacle['home_prob'] = pinnacle['home_odds'].apply(odds_to_prob)
            pinnacle['away_prob'] = pinnacle['away_odds'].apply(odds_to_prob)
            print(f'\nPinnacle lines: {len(pinnacle)}')
            print(pinnacle[['home_team', 'away_team', 'home_prob', 'away_prob']].head(10))
    
    con2.close()
else:
    print('No collector database yet - run collector worker first')

## Edge Distribution After Fees

In [ ]:
# Simulate edge calculations across price levels
edge_calc = EdgeCalculator(fee_calc)

# For each market price, compute edge at various model probabilities
market_prices = np.arange(0.10, 0.91, 0.05)
model_offsets = [-0.10, -0.05, -0.03, 0.03, 0.05, 0.10]

print('Fee-Adjusted Edge (positive = profitable):')
print(f'{"Market":>8} | ' + ' | '.join(f'Model+{o:+.0%}' for o in model_offsets))
print('-' * 80)
for mp in market_prices:
    edges = []
    for offset in model_offsets:
        model_p = mp + offset
        if 0 < model_p < 1:
            result = edge_calc.compute_edge(
                model_prob=model_p,
                market_price=Decimal(str(round(mp, 2))),
            )
            edges.append(f'{float(result.net_edge):+.3f}')
        else:
            edges.append('  N/A  ')
    print(f'{mp:8.2f} | ' + ' | '.join(f'{e:>10}' for e in edges))

## Execution Analysis (if available)

In [ ]:
EXEC_DB = DATA_DIR / 'execution.duckdb'
if EXEC_DB.exists():
    con3 = duckdb.connect(str(EXEC_DB), read_only=True)
    tables = con3.execute("SHOW TABLES").fetchall()
    print(f'Execution DB tables: {[t[0] for t in tables]}')
    
    # Load predictions if available
    try:
        df_preds = con3.execute("""
            SELECT * FROM predictions
            ORDER BY prediction_time DESC
            LIMIT 100
        """).df()
        print(f'\nRecent predictions: {len(df_preds)}')
        if len(df_preds) > 0:
            print(f'Probability range: {df_preds["probability"].min():.3f} - {df_preds["probability"].max():.3f}')
            print(df_preds.head())
    except Exception as e:
        print(f'No predictions table yet: {e}')
    
    con3.close()
else:
    print('No execution database yet - run execution worker first')

In [ ]:
con.close()
print('Done.')